# مطابقة البنك مع دفتر الأستاذ · أكتوبر 2026

دفتر معدّ مسبقًا لشركة ظِلال (بيانات اصطناعية). شغّل الخلايا بالترتيب: لا تحتاج إلى كتابة أي رموز.
الدفتر **يحسب** ويثبت حسابه بمجاميع رقابية، و**يقترح** تصنيفًا للبنود غير المتطابقة. القرار في كل بند لك.

In [ ]:
# الإعداد: يجد الملفات بجوار الدفتر أو في المجلد الأعلى. في خدمة دفاتر مستضافة: ارفع الملفات بجوار الدفتر.
import csv, datetime, hashlib
from pathlib import Path
import pandas as pd

def find(name):
    for p in (Path("."), Path(".."), Path("files")):
        if (p / name).exists():
            return p / name
    raise FileNotFoundError(name + " : ضع الملف بجوار الدفتر")

def log_run(notebook, inputs, totals):
    """يضيف سطرًا إلى سجل التشغيل: متى، وعلى أي ملفات (بصمتها)، وبأي مجاميع رقابية."""
    stamp = datetime.datetime.now().isoformat(timespec="seconds")
    marks = ";".join(p.name + ":" + hashlib.sha256(p.read_bytes()).hexdigest()[:12] for p in inputs)
    new = not Path("run_log.csv").exists()
    with open("run_log.csv", "a", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["run_at", "notebook", "inputs_sha256", "control_totals"])
        w.writerow([stamp, notebook, marks, totals])
    print("سُجّل التشغيل في run_log.csv")

## 1 · المجاميع الرقابية قبل أي معالجة

In [ ]:
OPENING = 2350000.00
bank_path, ledger_path = find("bank_statement_oct.csv"), find("ledger_cash_oct.csv")
bank = pd.read_csv(bank_path, dtype={"reference": str})
ledger = pd.read_csv(ledger_path, dtype={"reference": str})
for df in (bank, ledger):
    df["h"] = (df["amount"] * 100).round().astype("int64")      # المبلغ بالهللات: لا كسور عشرية في المقارنة
    df["key"] = df["reference"].str.strip().str.upper()         # مفتاح الربط: المرجع بعد التنظيف
control_in = {"bank_lines": len(bank), "bank_sum": bank["h"].sum() / 100,
              "ledger_lines": len(ledger), "ledger_sum": ledger["h"].sum() / 100}
print(control_in)

## 2 · المطابقة بالمرجع ثم المبلغ

كل سطر يُستخدم مرة واحدة: السطر الثاني بالمرجع والمبلغ نفسيهما لا يجد شريكًا ويبقى ظاهرًا.

In [ ]:
for df in (bank, ledger):
    df["n"] = df.groupby(["key", "h"]).cumcount()
m = bank.merge(ledger, on=["key", "h", "n"], how="outer", suffixes=("_bank", "_ledger"), indicator=True)
matched = m[m["_merge"] == "both"]
bank_only = m[m["_merge"] == "left_only"]
ledger_only = m[m["_merge"] == "right_only"]
control_out = {"matched_pairs": len(matched), "matched_sum": matched["h"].sum() / 100,
               "bank_unmatched": len(bank_only), "ledger_unmatched": len(ledger_only)}
print(control_out)
# الفحص الرقابي: لا سطر ضاع ولا سطر استُخدم مرتين
assert len(matched) + len(bank_only) == len(bank), "أسطر البنك لا تتطابق عددًا"
assert len(matched) + len(ledger_only) == len(ledger), "أسطر الدفتر لا تتطابق عددًا"
assert matched["h"].sum() + bank_only["h"].sum() == bank["h"].sum(), "مجموع البنك لا يتطابق"
assert matched["h"].sum() + ledger_only["h"].sum() == ledger["h"].sum(), "مجموع الدفتر لا يتطابق"
print("المجاميع الرقابية متفقة: متطابق + غير متطابق = كل ملف، عددًا ومبلغًا")

## 3 · البنود غير المتطابقة وتصنيف مقترح

التصنيف اقتراح من قاعدة مكتوبة أدناه. اقرأ القاعدة، ثم احكم على كل بند من مستنده.

In [ ]:
bank_keys, ledger_keys = set(bank["key"]), set(ledger["key"])
pairs = set(zip(matched["key"], matched["h"]))

def suggest_ledger(r):
    if r["key"] not in bank_keys:
        return "مدفوع لم يُصرف بعد" if r["h"] < 0 else "إيداع بالطريق"
    return "قيد مكرر" if (r["key"], r["h"]) in pairs else "فرق مبلغ"

def suggest_bank(r):
    return "فرق مبلغ" if r["key"] in ledger_keys else "في البنك ولم يُقيَّد"

lo = ledger_only.assign(side="ledger", id=ledger_only["entry_no"], date=ledger_only["date_ledger"],
                        description=ledger_only["description_ledger"], suggested=ledger_only.apply(suggest_ledger, axis=1))
bo = bank_only.assign(side="bank", id=bank_only["line_no"], date=bank_only["date_bank"],
                      description=bank_only["description_bank"], suggested=bank_only.apply(suggest_bank, axis=1))
unmatched = pd.concat([bo, lo])[["side", "id", "date", "key", "description", "h", "suggested"]]
unmatched["amount"] = unmatched["h"] / 100
print(unmatched.groupby(["side", "suggested"])["amount"].agg(["count", "sum"]).round(2))
unmatched.drop(columns="h").to_csv("unmatched_suggested.csv", index=False, encoding="utf-8-sig")

## 4 · قائمة المطابقة

الرصيدان المعدَّلان يجب أن يتساويا. إن لم يتساويا فهناك بند لم يُصنَّف صحيحًا.

In [ ]:
def total(side, label):
    x = unmatched[(unmatched["side"] == side) & (unmatched["suggested"] == label)]
    return len(x), x["h"].sum() / 100

bank_closing = round(OPENING + control_in["bank_sum"], 2)
ledger_closing = round(OPENING + control_in["ledger_sum"], 2)
n_dep, dep = total("ledger", "إيداع بالطريق")
n_out, out = total("ledger", "مدفوع لم يُصرف بعد")
n_dup, dup = total("ledger", "قيد مكرر")
n_bo, bo_sum = total("bank", "في البنك ولم يُقيَّد")
_, err_bank = total("bank", "فرق مبلغ")
_, err_ledger = total("ledger", "فرق مبلغ")
adjusted_bank = round(bank_closing + dep + out, 2)
adjusted_ledger = round(ledger_closing - dup + bo_sum + (err_bank - err_ledger), 2)
statement = {"bank_closing": bank_closing, "deposits_in_transit": (n_dep, dep), "outstanding_payments": (n_out, out),
             "adjusted_bank": adjusted_bank, "ledger_closing": ledger_closing, "duplicates": (n_dup, dup),
             "bank_only": (n_bo, bo_sum), "amount_correction": round(err_bank - err_ledger, 2), "adjusted_ledger": adjusted_ledger}
for k, v in statement.items():
    print(f"{k:22} {v}")
print("الرصيدان المعدَّلان متساويان:", adjusted_bank == adjusted_ledger)

In [ ]:
log_run("reconcile", [bank_path, ledger_path],
        f"bank {control_in['bank_lines']}/{control_in['bank_sum']:.2f}; ledger {control_in['ledger_lines']}/{control_in['ledger_sum']:.2f}; "
        f"matched {control_out['matched_pairs']}; adjusted {adjusted_bank:.2f}/{adjusted_ledger:.2f}")